# **External Ballistics and Ray Propagation**
## A Numerically Stable Trajectory Engine for Nonlinear ODE Systems

**Author:** Vanya Videva  
**Course:** Math Concepts for Developers 

**Date:** March 2026

---

## 1. Introduction

#### **What is External Ballistics?**
External ballistics is the study of a projectile's motion **after** it leaves the barrel — subject to gravity, aerodynamic drag, and atmospheric conditions. 
The mathematical foundation dates back to Newton's laws of motion, and today it is essential in applications ranging from sport shooting to game development.

The simplest model treats the projectile as a **point mass** moving through the air, described by a system of nonlinear ordinary differential equations (ODEs).

#### **Connection to Ray Propagation**
A ray of light traveling through a medium with a varying refractive index follows a curved path — described by the **same mathematical structure** as a ballistic trajectory. Both systems are nonlinear ODE problems, making them ideal for demonstrating a unified numerical engine.

#### **Motivation**
The choice of numerical integrator has a significant impact on simulation accuracy. 
A poorly chosen method leads to:
- **Energy drift** — the simulation gains or loses energy artificially
- **Numerical instability** — small errors grow exponentially over time

This project explores and compares three integrators to demonstrate these effects.

#### **Objectives**
1. Implement a trajectory engine in **Python** using NumPy and Matplotlib
2. Model two physical systems: **ballistics with drag** and **ray in gradient medium**
3. Compare three numerical integrators: **Explicit Euler, Symplectic Euler, RK4**
4. Analyze **numerical stability** and **energy conservation** across methods
5. Visualize trajectory, energy error, and stability as a function of time step $\Delta t$



**Keywords:** External ballistics, Trajectory simulation, Numerical methods, ODE systems, Numerical stability, Ray propagation, Runge-Kutta, Symplectic Euler

## 2. Mathematical Framework

#### **What is an ODE System?**

An **Ordinary Differential Equation (ODE)** describes how a quantity changes over time. For a moving projectile, tracking two quantities:

- **Position** — where is the projectile? $(x, y)$
- **Velocity** — how fast and in which direction? $(v_x, v_y)$

The state of the system at any time $t$ is described by a vector:

$$\mathbf{s} = (x, y, v_x, v_y)$$

The ODE system describes how this state evolves over time:

$$\frac{d\mathbf{s}}{dt} = f(\mathbf{s}, t)$$

In plain words: *"the rate of change of the state equals some function 
of the current state"*

#### **Connection to Derivatives**

As shown by Darakchiev (2026), instantaneous velocity is defined as the limit of the difference quotient (the ratio of change in position to change in time) as $\Delta t \to 0$:

$$v(t) = \lim_{\Delta t \to 0} \frac{s(t + \Delta t) - s(t)}{\Delta t}$$

This means velocity is the **first derivative** of position, and acceleration is the **first derivative** of velocity:

$$v_x = \frac{dx}{dt}, \quad v_y = \frac{dy}{dt}$$

$$a_x = \frac{dv_x}{dt}, \quad a_y = \frac{dv_y}{dt}$$

This gives complete ODE system:

$$\frac{dx}{dt} = v_x, \quad \frac{dy}{dt} = v_y$$

$$\frac{dv_x}{dt} = a_x, \quad \frac{dv_y}{dt} = a_y$$

Where $a_x$ and $a_y$ are accelerations caused by the forces acting on the projectile — which will be defined in the next section.

## 3. Forces Acting on the Projectile

#### **Gravity**

The simplest force acting on the projectile is gravity - a constant downward acceleration:

$$a_x^{gravity} = 0, \quad a_y^{gravity} = -g$$

Where $g = 9.81 \, m/s^2$ is the gravitational acceleration. 
It acts only in the vertical direction - no horizontal component.

#### **Aerodynamic Drag**

In reality, the proejctile also experiences air resistance. The drag force acts opposite to the velocity direction and is proportional to the square of the speed:

$$F_{drag} = \frac{1}{2} \rho C_d A v^2$$

Where:
- $\rho$ - air density ($1.225 \, kg/m^3$ at sea level)
- $C_d$ - drag coefficient, depends on the projectile shape
- $A$ - cross-sectional area of the projectile
- $v = \sqrt{v_x^2 + v_y^2}$ - speed (magnitude of velocity)

This is called **quadratic drag** - the faster the projectile, the stronger the resistance.

#### **Total Acceleration**

Combining both forces using Newton's second law $F = ma$, rearranged to $a = F/m$:

$$a_x = -\frac{F_{drag}}{m} \cdot \frac{v_x}{v}$$

$$a_y = -g - \frac{F_{drag}}{m} \cdot \frac{v_y}{v}$$

The $\frac{v_x}{v}$ and $\frac{v_y}{v}$ terms give the direction of the drag - always opposite to the direction of motion. This completes the ODE system.

#### **Ray Propagation - A Different Kind of Force**

For a light ray traveling through a medium with varying refractive index $n(x, y)$, there is no gravity or drag, but the ray still bends. The ray curves toward regions of higher refractive index, described by:

$$\frac{d^2\mathbf{r}}{dt^2} = \nabla n(\mathbf{r})$$

Where $\nabla n$ is the gradient of the refractive index (Darakchiev, 2026) — pointing in the direction of greatest change.

This is mathematically identical in structure to the ballistics ODE system - just with different physical forces. A unified numerical engine can therefore simulate both systems with the same code.


## 4. Numerical Integration Methods

The ODE systems derived in the previous sections generally do not have simple analytical solutions once nonlinear forces such as quadratic drag are included. Instead, the trajectory must be approximated numerically by evaluating the derivatives over small time steps.

A numerical integrator advances the system state forward in time using the relation 

$$\mathbf{s}_{n+1} = \mathbf{s}_n + \Delta t$$

where $\mathbf{s}_n$ is the system state at time $t_n$ and $\Delta s$ is computed from derivatives of the system.

The choice of numerical method strongly affects simulation accuracy, stability, and energy behavior, In this project, three commonly used integrators are compared:

- "Explicit Euler"
- "Symplectic Euler"
- "Runge–Kutta 4 (RK4)"

#### **Explicit Euler Method**

The Explicit Euler method is the simplest numerical integrator. It approximates the system’s evolution by assuming the derivative remains constant over a small time step $\Delta t$

$$\mathbf{s}_{n+1} = \mathbf{s}_n + \Delta t\, f(\mathbf{s}_n, t_n)$$

In the projectile system this means the position and velocity are updated using the current acceleration and velocity values.
The Explicit Euler method is fast, but can be unstable for large time steps $\Delta t$.

#### **Symplectic Euler Method**

A small but important modification of Explicit Euler method — updates velocity first, then uses the **new** velocity to update position:

$$v_{n+1} = v_n + \Delta t\, a(x_n, v_n)$$
$$x_{n+1} = x_n + \Delta t\, v_{n+1}$$

This preserves important geometric properties of the system and produces better long-term energy behavior in physical simulations (Cook, 2026).

#### **Runge-Kutta 4 Method (RK4)**

The fourth-order Runge-Kutta method improves accuracy by evaluating the derivative at four intermediate points within each timestep:

$$k_1 = f(\mathbf{s}_n, t_n)$$
$$k_2 = f\left(\mathbf{s}_n + \frac{\Delta t}{2}k_1,\ t_n + \frac{\Delta t}{2}\right)$$
$$k_3 = f\left(\mathbf{s}_n + \frac{\Delta t}{2}k_2,\ t_n + \frac{\Delta t}{2}\right)$$
$$k_4 = f(\mathbf{s}_n + \Delta t,\ k_3,\ t_n + \Delta t)$$

$$\mathbf{s}_{n+1} = \mathbf{s}_n + \frac{\Delta t}{6}(k_1 + 2k_2 + 2k_3 + k_4)$$

RK4 is more accurate than Euler methods, though at the cost of four derivative evaluations per timestep.


---


**In the following implementation, these three methods will be applied to both the ballistic trajectory and the ray propagation system. Their performance will be compared in terms of:**

* **trajectory accuracy**
* **numerical stability**
* **energy conservation**
* **sensitivity to timestep $\Delta t$**
```
